# LSTM Model (VCB)

Single target: **5-day return** (regression). Up/down direction is **not** a model target — it is evaluated *from the return prediction* (sign for dir_acc; predicted-return value as the score for AUC), reported alongside R2.

## Import Libraries

In [9]:
import json
import os

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import lightning as L
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, TensorDataset

## Parameters

In [10]:
DATASET_NAME = "vcb_lb20_h5_f200_dynta_tr70_val15_test15_std"
DATA_DIR = os.path.join("../../train_test_set", DATASET_NAME)

CLIP_VALUE = 10.0
USE_TOP_FEATURES = 80     # best single-stock setting

# --- model (small + regularized) ---
HIDDEN_SIZE = 48
NUM_LAYERS = 1
DROPOUT = 0.4

# --- training ---
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-3
EPOCHS = 100
PATIENCE = 12
GRAD_CLIP = 1.0
RANDOM_STATE = 42

L.seed_everything(RANDOM_STATE, workers=True)

Seed set to 42


42

## Load Data

In [11]:
def load(name):
    return np.load(os.path.join(DATA_DIR, name))

X_train, y_train = load("X_train.npy"), load("y_train.npy")
X_val, y_val = load("X_val.npy"), load("y_val.npy")
X_test, y_test = load("X_test.npy"), load("y_test.npy")

with open(os.path.join(DATA_DIR, "metadata.json")) as f:
    metadata = json.load(f)
target_scaler = joblib.load(os.path.join(DATA_DIR, "target_scaler.pkl"))

if USE_TOP_FEATURES:
    feat_cols = metadata["feature_columns"]
    ranking = pd.read_csv(os.path.join(DATA_DIR, "feature_ranking.csv"))
    ranked = ranking[ranking["feature"].isin(feat_cols)].sort_values("blended_score", ascending=False)
    idx = [feat_cols.index(c) for c in ranked["feature"].head(USE_TOP_FEATURES)]
    X_train, X_val, X_test = X_train[:, :, idx], X_val[:, :, idx], X_test[:, :, idx]
    sel_names = [feat_cols[i] for i in idx]
    n_macro = sum(n.startswith(('economy_', 'bonds_')) for n in sel_names)
    print(f"Subset to {len(idx)} features  (macro:{n_macro} TA/price:{len(idx)-n_macro})")

N_FEATURES = X_train.shape[-1]
print(f"X_train {X_train.shape}  X_val {X_val.shape}  X_test {X_test.shape}")

Subset to 80 features  (macro:7 TA/price:73)
X_train (2926, 20, 80)  X_val (631, 20, 80)  X_test (632, 20, 80)


## DataModule

In [12]:
class StockDataModule(L.LightningDataModule):
    def __init__(self, splits, batch_size, clip_value):
        super().__init__()
        self.splits = splits
        self.batch_size = batch_size
        self.clip_value = clip_value

    def _ds(self, key):
        X, y = self.splits[key]
        X = np.clip(X, -self.clip_value, self.clip_value)
        return TensorDataset(torch.from_numpy(X).float(), torch.from_numpy(y).float().unsqueeze(-1))

    def setup(self, stage=None):
        self.train_ds, self.val_ds, self.test_ds = self._ds("train"), self._ds("val"), self._ds("test")

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.val_ds, batch_size=self.batch_size, shuffle=False)

    def test_dataloader(self):
        return DataLoader(self.test_ds, batch_size=self.batch_size, shuffle=False)


splits = {"train": (X_train, y_train), "val": (X_val, y_val), "test": (X_test, y_test)}
datamodule = StockDataModule(splits, BATCH_SIZE, CLIP_VALUE)

## Model

In [13]:
class LSTMRegressor(L.LightningModule):
    """Bidirectional LSTM + attention pooling over timesteps -> linear head -> scalar return.

    Single target (return). Huber loss is robust to fat-tailed big-move returns.
    """

    def __init__(self, num_features, hidden_size, num_layers, dropout, lr, weight_decay):
        super().__init__()
        self.save_hyperparameters()
        self.lstm = nn.LSTM(num_features, hidden_size, num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0, bidirectional=True)
        d = hidden_size * 2
        self.attn = nn.Linear(d, 1)
        self.head = nn.Sequential(nn.Linear(d, d // 2), nn.ReLU(), nn.Dropout(dropout), nn.Linear(d // 2, 1))
        self.criterion = nn.SmoothL1Loss(beta=1.0)

    def forward(self, x):
        out, _ = self.lstm(x)
        w = torch.softmax(self.attn(out), dim=1)
        ctx = (w * out).sum(dim=1)
        return self.head(ctx)

    def _step(self, batch, stage):
        x, y = batch
        loss = self.criterion(self(x), y)
        self.log(f"{stage}_loss", loss, prog_bar=True, on_epoch=True, on_step=False)
        return loss

    def training_step(self, b, _):
        return self._step(b, "train")
    def validation_step(self, b, _):
        return self._step(b, "val")
    def test_step(self, b, _):
        return self._step(b, "test")
    def predict_step(self, b, _):
        return self(b[0])

    def configure_optimizers(self):
        opt = torch.optim.Adam(self.parameters(), lr=self.hparams.lr, weight_decay=self.hparams.weight_decay)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=5)
        return {"optimizer": opt, "lr_scheduler": {"scheduler": sched, "monitor": "val_loss"}}


model = LSTMRegressor(N_FEATURES, HIDDEN_SIZE, NUM_LAYERS, DROPOUT, LEARNING_RATE, WEIGHT_DECAY)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}")

Trainable parameters: 54,722


## Train

In [14]:
early_stop = EarlyStopping(monitor="val_loss", mode="min", patience=PATIENCE)
checkpoint = ModelCheckpoint(dirpath="checkpoints", filename=f"lstm_{DATASET_NAME}",
                             monitor="val_loss", mode="min", save_top_k=1)
logger = CSVLogger(save_dir=".", name="lightning_logs")

trainer = L.Trainer(max_epochs=EPOCHS, accelerator="auto", devices=1, gradient_clip_val=GRAD_CLIP,
                    callbacks=[early_stop, checkpoint], logger=logger, log_every_n_steps=10,
                    enable_progress_bar=True)
trainer.fit(model, datamodule=datamodule)
print(f"Best val_loss = {float(checkpoint.best_model_score):.4f}")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
d:\GIT\master-thesis\mt_env\Lib\site-packages\lightning\pytorch\callbacks\model_checkpoint.py:881: Checkpoint directory D:\GIT\master-thesis\src\model\lstm\checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type         | Params | Mode  | FLOPs
-----------------------------------------------------------
0 | lstm      | LSTM         | 49.9 K | train | 0    
1 | attn      | Linear       | 97     | train | 0    
2 | head      | Sequential   | 4.7 K  | train | 0    
3 | criterion | SmoothL1Loss | 0      | train | 0    
-----------------------------------------------------------
54.7 K    Trainable params
0         Non-trainable params
54.7 K

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

d:\GIT\master-thesis\mt_env\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.
d:\GIT\master-thesis\mt_env\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Best val_loss = 0.2760


## Evaluate

Return metrics (R2/RMSE/corr) plus direction metrics **derived from the return prediction**: dir_acc = sign agreement; AUC = how well the predicted return ranks up-days above down-days.

In [15]:
best_model = LSTMRegressor.load_from_checkpoint(checkpoint.best_model_path)

def report(name, X, y):
    Xc = np.clip(X, -CLIP_VALUE, CLIP_VALUE)
    loader = DataLoader(TensorDataset(torch.from_numpy(Xc).float()), batch_size=BATCH_SIZE)
    ret = torch.cat(trainer.predict(best_model, loader)).numpy().ravel()
    p = target_scaler.inverse_transform(ret.reshape(-1, 1)).ravel()      # predicted return (real units)
    t = target_scaler.inverse_transform(y.reshape(-1, 1)).ravel()        # actual return (real units)
    # return metrics
    ss_res = float(np.sum((t - p) ** 2)); ss_tot = float(np.sum((t - t.mean()) ** 2))
    r2 = 1.0 - ss_res / ss_tot
    rmse = float(np.sqrt(np.mean((p - t) ** 2)))
    corr = float(np.corrcoef(p, t)[0, 1])
    # direction metrics derived from the SAME return prediction
    up = (t > 0).astype(int)
    dir_acc = float(np.mean(np.sign(p) == np.sign(t)))
    auc = float(roc_auc_score(up, p)) if len(np.unique(up)) > 1 else float('nan')
    print(f"{name:5s} | R2={r2:+.4f} RMSE={rmse:.3f} corr={corr:+.3f} | dir_acc={dir_acc:.3f} AUC={auc:.3f}")
    return p, t

print("target=return; direction (dir_acc/AUC) derived from the return prediction")
report("train", X_train, y_train)
report("val", X_val, y_val)
test_pred, test_true = report("test", X_test, y_test)

d:\GIT\master-thesis\mt_env\Lib\site-packages\lightning\fabric\utilities\cloud_io.py:73: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


target=return; direction (dir_acc/AUC) derived from the return prediction


d:\GIT\master-thesis\mt_env\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.


Predicting: |          | 0/? [00:00<?, ?it/s]

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


...

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


val   | R2=-0.0160 RMSE=3.612 corr=+0.049 | dir_acc=0.532 AUC=0.507


d:\GIT\master-thesis\mt_env\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.


Predicting: |          | 0/? [00:00<?, ?it/s]

test  | R2=-0.0060 RMSE=3.593 corr=+0.118 | dir_acc=0.473 AUC=0.526


## Save Model

In [16]:
print(f"Best checkpoint: {checkpoint.best_model_path}")
print(f"Best val_loss:   {float(checkpoint.best_model_score):.4f}")
print(f"CSV logs:        {logger.log_dir}")

Best checkpoint: D:\GIT\master-thesis\src\model\lstm\checkpoints\lstm_vcb_lb20_h5_f200_dynta_tr70_val15_test15_std-v9.ckpt
Best val_loss:   0.2760
CSV logs:        .\lightning_logs\version_11
